 # Décorateurs

Un décorateur est une fonction qui manipule une fonction ou une classe. C'est un élément du langage Python très pratique.

 Nous allons voir dans ces travaux pratiques comment les utiliser.

## Décorateurs de fonctions : fonctions qui manipulent des fonctions

Écrivez une fonction `squared`, qui prend en entrée une fonction et qui renvoie cette même fonction dont le résultat est passé au carré :

- la fonction originale prend toujours deux arguments
- la fonction originale produit toujours un nombre
- la fonction squared prend en entrée et produit en sortie **une fonction**

In [ ]:
def add(a, b):
  return a + b


def sub(a, b):
  return a - b


def mul(a, b):
  return a * b


def squared(function):
  return function  # Votre code ici


print(squared(add)(2, 2))  # Doit valoir 16
print(squared(sub)(10, 5))  # Doit valoir 25
print(squared(mul)(3, 4))  # Doit valoir 144

### Solution

In [ ]:
def squared(function):
  def new_function(a, b):
    return function(a, b) ** 2
  return new_function


print(squared(add)(2, 2))
print(squared(sub)(10, 5))
print(squared(mul)(3, 4))

## Ordre des décorateurs empilés

Les deux décorateurs ci-dessous affichent un message avant et après l'appel de la fonction qu'ils décorent, et entourent son résultat d'une balise HTML.

Sans exécuter de code, écrivez dans l'ordre les lignes qu'affichera `print(greet("Ada"))`. Vérifiez ensuite votre réponse en exécutant les deux cellules.

In [ ]:
import functools


def bold(function):
    @functools.wraps(function)
    def new_function(*args, **kwargs):
        print("bold: avant")
        result = function(*args, **kwargs)
        print("bold: après")
        return f"<b>{result}</b>"

    return new_function


def italic(function):
    @functools.wraps(function)
    def new_function(*args, **kwargs):
        print("italic: avant")
        result = function(*args, **kwargs)
        print("italic: après")
        return f"<i>{result}</i>"

    return new_function


@bold
@italic
def greet(name):
    print("greet")
    return f"Bonjour {name}"

In [ ]:
print(greet("Ada"))

*Votre réponse*

### Solution

```text
bold: avant
italic: avant
greet
italic: après
bold: après
<b><i>Bonjour Ada</i></b>
```

`@bold @italic def greet` équivaut à `greet = bold(italic(greet))`&nbsp;: `bold` enveloppe `italic`, qui enveloppe `greet`. L'appel traverse les couches de l'extérieur vers l'intérieur, et le résultat ressort dans l'ordre inverse.

## Mesure du temps d'exécution d'une fonction

Créez un décorateur `timeit` qui affiche le temps d'exécution d'une fonction. Vous pourrez vous aider de la fonction [`time.perf_counter`](https://docs.python.org/fr/3/library/time.html#time.perf_counter)

In [ ]:
# Votre code ici

### Solution

In [ ]:
import time


def timeit(f):
  def new_f(*args, **kwargs):
    start = time.perf_counter()
    result = f(*args, **kwargs)
    print(f"Time elapsed during '{f.__name__}' call: {time.perf_counter() - start}s")
    return result
  return new_f


@timeit
def add(a, b):
  return a + b


add(1, 2)

## Enregistreur de classe

Il est souvent intéressant d'enregistrer des classes ou fonctions dans une structure de données (un registre), pour par exemple proposer un système de plugins.

Créez un décorateur de classe qui enregistre chaque classe à laquelle il est appliqué dans un dictionnaire, sous la clef `classe.__name__`.

In [ ]:
# Votre code ici

En plus d'ajouter le décorateur aux classes que vous souhaitez enregistrer, quelle étape importante est nécessaire pour que l'enregistrement se fasse ?

*Votre réponse*

### Solution

Ce pattern, appelé *Registry* en anglais, permet à une librairie d'enregistrer du code client pour pouvoir ensuite l'exécuter à un moment approprié.

Cela se passe souvent en trois étapes :

1. Mise à disposition d'un décorateur pour enregistrer une fonction ou une classe dans le code client
2. Utilisation du décorateur côté client
3. Utilisation des éléments enregistrés côté librairie

Par exemple, pour l'étape 1. :

In [ ]:
from typing import Protocol


class SupportsProcess(Protocol):
  def process(self) -> None: ...


_plugins = {}


def register[T: SupportsProcess](cls: type[T]) -> type[T]:
  _plugins[cls.__name__] = cls
  return cls

Puis pour l'étape 2. :

In [ ]:
@register
class CustomClientProcessor:
  def process(self) -> None:
    print("Print depuis la méthode process custom du client")

Attention, il est nécessaire de charger le module qui contient la classe (sinon le décorateur ne sera pas exécuté et la classe ne sera pas enregistrée).

Enfin, pour l'étape 3. :

In [ ]:
for name, cls in _plugins.items():
  print(f"Appel de la méthode process de {name}")
  cls().process()

## Définition d'un décorateur avec argument

Reprenez l'exercice précédent mais cette fois le décorateur doit récupérer un argument `name` à utiliser comme clef de registre à la place de `classe.__name__`.

In [ ]:
# Votre code ici

### Solution

In [ ]:
from collections.abc import Callable

_plugins = {}


def register[T](name: str) -> Callable[[type[T]], type[T]]:
    def decorator(cls: type[T]) -> type[T]:
        _plugins[name] = cls
        return cls

    return decorator


@register("csv")
class CsvExporter:
    pass


_plugins

## Utilisation de [`functools.wraps`](https://docs.python.org/fr/3/library/functools.html#functools.wraps)

Essayez de manipuler les fonctions modifiées dans le reste du TP et de trouver leur nom (attribut `__name__`) ou encore leur doc (attribut `__doc__`). Que remarquez-vous ?

*Votre réponse*

Créez un décorateur `timeit` qui prend un argument `unit` (`"s"` ou `"ms"`), l'unité dans laquelle afficher le temps d'exécution, puis utilisez `functools.wraps` pour régler ce problème.

In [ ]:
# Votre code ici

### Solution

In [ ]:
import functools
import time


def timeit(unit="s"):
  def decorator(function):
    @functools.wraps(function)
    def new_f(*args, **kwargs):
      start = time.perf_counter()
      result = function(*args, **kwargs)
      duration = time.perf_counter() - start
      if unit == "ms":
        duration *= 1000
      elif unit != "s":
        raise ValueError("Can only use s or ms as unit argument")
      print(f"Time elapsed during '{function.__name__}' call: "
            f"{duration}{unit}")
      return result
    return new_f
  return decorator


@timeit(unit="ms")
def add(a, b):
  """Add two integers."""
  return a + b


print(add.__name__, add.__doc__)

## Mise en cache des résultats

Écrivez un décorateur `memoize` qui mémorise dans un dictionnaire le résultat de chaque appel, avec ses arguments pour clef, et le renvoie directement quand la fonction est rappelée avec les mêmes arguments.

Appliquez-le à `fibonacci`, puis comparez avec votre décorateur `timeit` la durée de `fibonacci(32)` sans cache, avec `memoize`, et avec [`functools.cache`](https://docs.python.org/fr/3/library/functools.html#functools.cache).

In [ ]:
def fibonacci(n: int) -> int:
    return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)

In [ ]:
# Votre code ici

### Solution

In [ ]:
import functools


def memoize(function):
    cache = {}

    @functools.wraps(function)
    def new_function(*args):
        if args not in cache:
            cache[args] = function(*args)
        return cache[args]

    return new_function


def fibonacci_plain(n: int) -> int:
    return n if n < 2 else fibonacci_plain(n - 1) + fibonacci_plain(n - 2)


@memoize
def fibonacci(n: int) -> int:
    return n if n < 2 else fibonacci(n - 1) + fibonacci(n - 2)


@functools.cache
def fibonacci_cache(n: int) -> int:
    return n if n < 2 else fibonacci_cache(n - 1) + fibonacci_cache(n - 2)


for f in (fibonacci_plain, fibonacci, fibonacci_cache):
    timeit(unit="ms")(f)(32)

Sans cache, `fibonacci(32)` recalcule les mêmes valeurs des millions de fois&nbsp;; avec, chaque valeur n'est calculée qu'une fois. `functools.cache` fait la même chose que `memoize`, en gérant aussi les arguments nommés, et fournit `cache_info()` et `cache_clear()`.